# 05 — Model Evaluation
## SustAInable | Neighborhood Heat Illness Risk Prediction

---

**Purpose:** Full evaluation of the trained SustAInable model. This notebook answers the question that matters to every stakeholder — public health departments, emergency managers, fellowship reviewers, and potential partners:

> *Does this model actually work? And is it better than what cities are already using?*

**Inputs:**
- `outputs/sustainable_model.json` ← trained model from Notebook 04
- `outputs/sustainable_model_metadata.json` ← threshold, features, config from Notebook 04
- `data/processed/sustainable_labeled.csv` ← labeled dataset from Notebook 03

**Outputs:**
- `outputs/05a_roc_curve.png`
- `outputs/05b_confusion_matrix.png`
- `outputs/05c_score_distribution.png`
- `outputs/05d_per_event_performance.png`
- `outputs/05e_calibration_curve.png`
- `outputs/05f_risk_tier_map.png`

---

> ⚠️ **Run order matters.**
> This notebook requires Notebooks 02, 03, and 04 to have been run first,
> and requires `xgboost`, `scikit-learn`, and `imbalanced-learn` to be installed.
>
> If you have not installed them yet, run this first:
> ```
> pip install xgboost scikit-learn imbalanced-learn
> ```
> Then run Notebook 04 to generate the model file before opening this one.

**Author:** Nico Meyering, MPA | Equitech Futures Civic Tech Institute, CTI 2026

---

## Step 0: Imports, Dependency Check, and Preflight

This cell checks everything before running a single line of evaluation logic. If anything is missing, it tells you exactly what to do and stops cleanly rather than crashing mid-notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

READY = True

# ── Dependency check ───────────────────────────────────────────────────────────
print("Checking dependencies...")
missing_pkgs = []

try:
    import xgboost as xgb
    print(f"  ✅ xgboost          {xgb.__version__}")
except ImportError:
    missing_pkgs.append("xgboost")
    print("  ❌ xgboost          — not installed")
    READY = False

try:
    import sklearn
    from sklearn.metrics import (
        roc_auc_score, roc_curve, recall_score, precision_score,
        f1_score, confusion_matrix, precision_recall_curve,
        average_precision_score, brier_score_loss
    )
    from sklearn.calibration import calibration_curve
    print(f"  ✅ scikit-learn     {sklearn.__version__}")
except ImportError:
    missing_pkgs.append("scikit-learn")
    print("  ❌ scikit-learn     — not installed")
    READY = False

if missing_pkgs:
    print()
    print("🔴 Install missing packages, then re-run this cell:")
    print(f"   pip install {' '.join(missing_pkgs)}")
    print()
    print("After installing, also make sure Notebook 04 has been run")
    print("to generate the model file.")
else:
    print()
    print("✅ All dependencies satisfied.")

# ── File preflight ─────────────────────────────────────────────────────────────
DATA_PROCESSED = Path("../data/processed")
OUTPUTS        = Path("../outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)

LABELED_PATH  = DATA_PROCESSED / "sustainable_labeled.csv"
MODEL_PATH    = OUTPUTS / "sustainable_model.json"
METADATA_PATH = OUTPUTS / "sustainable_model_metadata.json"

print()
print("Checking required files...")
files = {
    "Labeled dataset (Notebook 03)": LABELED_PATH,
    "Trained model  (Notebook 04)": MODEL_PATH,
    "Model metadata (Notebook 04)": METADATA_PATH,
}
for label, path in files.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  ✅ {label:<35} ({size_kb:.1f} KB)")
    else:
        print(f"  🔴 {label:<35} NOT FOUND")
        READY = False

if not READY:
    print()
    print("🔴 PREFLIGHT FAILED — address the issues above before continuing.")
    print()
    print("Most likely fix: run Notebook 04 first.")
    print("It will generate sustainable_model.json and sustainable_model_metadata.json")
    print("in the outputs/ directory.")
else:
    print()
    print("✅ All files present. Ready to evaluate.")

## Step 1: Load Model, Metadata, and Labeled Data

In [ ]:
model = metadata = labeled_df = None
FEATURE_NAMES = []
FINAL_THRESHOLD = 0.30
IS_PROXY = False

if READY:
    # ── Load model ─────────────────────────────────────────────────────────────
    model = xgb.XGBClassifier()
    model.load_model(str(MODEL_PATH))
    print(f"✅ Model loaded from {MODEL_PATH.name}")

    # ── Load metadata ──────────────────────────────────────────────────────────
    with open(METADATA_PATH) as f:
        metadata = json.load(f)

    FEATURE_NAMES   = metadata.get("feature_names", [])
    FINAL_THRESHOLD = metadata.get("decision_threshold", 0.30)
    IS_PROXY        = metadata.get("label_mode", "real") == "proxy"

    print(f"✅ Metadata loaded")
    print(f"   Features:           {len(FEATURE_NAMES)}")
    print(f"   Decision threshold: {FINAL_THRESHOLD}")
    print(f"   Label mode:         {'⚠️  PROXY' if IS_PROXY else '✅  REAL (NSSP)'}")

    # ── Load labeled data ──────────────────────────────────────────────────────
    labeled_df = pd.read_csv(LABELED_PATH, dtype={'ZCTA': str})
    labeled_df['ZCTA'] = labeled_df['ZCTA'].str.zfill(5)

    # Align feature columns to what the model was trained on
    missing_feats = [f for f in FEATURE_NAMES if f not in labeled_df.columns]
    if missing_feats:
        print(f"\n⚠️  {len(missing_feats)} features from training not found in labeled data:")
        for f in missing_feats[:10]:
            print(f"   · {f}")
        print("   These will be filled with 0.")
        for f in missing_feats:
            labeled_df[f] = 0

    print(f"\n✅ Labeled data loaded: {len(labeled_df):,} rows × {labeled_df.shape[1]} columns")
    if IS_PROXY:
        print()
        print("⚠️  PROXY LABEL REMINDER")
        print("   All evaluation metrics in this notebook reflect proxy-labeled data.")
        print("   Numbers will change when real NSSP labels are used.")
else:
    print("⚠️  Load skipped — preflight failed. Resolve Step 0 issues first.")

## Step 2: Score All Observations

Run every ZCTA-event observation through the model to get probability scores. We score both the training set (sanity check) and holdout set (real evaluation).

In [ ]:
train_df = holdout_df = None
X_train = Y_train = X_holdout = Y_holdout = None
train_probs = holdout_probs = None
train_preds = holdout_preds = None

if model is not None and labeled_df is not None and FEATURE_NAMES:
    train_df   = labeled_df[labeled_df['split'] == 'train'].copy()
    holdout_df = labeled_df[labeled_df['split'] == 'holdout'].copy()

    # Fill any nulls with 0 before scoring (consistent with training)
    X_train   = train_df[FEATURE_NAMES].fillna(0)
    Y_train   = train_df['label']
    X_holdout = holdout_df[FEATURE_NAMES].fillna(0)
    Y_holdout = holdout_df['label']

    train_probs   = model.predict_proba(X_train)[:, 1]
    holdout_probs = model.predict_proba(X_holdout)[:, 1]
    train_preds   = (train_probs   >= FINAL_THRESHOLD).astype(int)
    holdout_preds = (holdout_probs >= FINAL_THRESHOLD).astype(int)

    # Attach scores back to dataframes for per-event analysis
    train_df   = train_df.copy()
    holdout_df = holdout_df.copy()
    train_df['prob_score']   = train_probs
    train_df['prediction']   = train_preds
    holdout_df['prob_score'] = holdout_probs
    holdout_df['prediction'] = holdout_preds

    print("✅ Scoring complete.")
    print(f"   Training rows scored:  {len(train_df):,}")
    print(f"   Holdout rows scored:   {len(holdout_df):,}")
    print()
    print(f"Training set (in-sample, sanity check):")
    t_rec = recall_score(Y_train, train_preds, zero_division=0)
    t_auc = roc_auc_score(Y_train, train_probs)
    print(f"  AUC-ROC: {t_auc:.4f}  |  Recall: {t_rec:.4f}")
    print()
    print(f"Holdout set (out-of-sample, the real number):")
    h_rec = recall_score(Y_holdout, holdout_preds, zero_division=0)
    h_auc = roc_auc_score(Y_holdout, holdout_probs)
    print(f"  AUC-ROC: {h_auc:.4f}  |  Recall: {h_rec:.4f}")

    gap = t_auc - h_auc
    if gap > 0.10:
        print(f"\n⚠️  AUC gap (train - holdout): {gap:.4f}")
        print("   Gap > 0.10 suggests some overfitting.")
        print("   Consider reducing max_depth or increasing regularization in Notebook 04.")
    else:
        print(f"\n✅ Train-holdout AUC gap: {gap:.4f}  — healthy generalization.")
else:
    print("⚠️  Scoring skipped — model or data not loaded.")

## Step 3: ROC Curve

The ROC curve plots true positive rate (recall) against false positive rate at every possible threshold. The area under the curve (AUC-ROC) is a threshold-independent measure of how well the model discriminates between elevated and non-elevated ZCTAs.

**Baseline comparison:** CDC HHI alone achieves AUC ≈ 0.72–0.74 (Kim et al. 2021; CDC Climate and Health Program). SustAInable must meaningfully outperform this to justify development.

In [ ]:
if holdout_probs is not None:
    fpr_h, tpr_h, _ = roc_curve(Y_holdout, holdout_probs)
    fpr_t, tpr_t, _ = roc_curve(Y_train,   train_probs)
    auc_holdout      = roc_auc_score(Y_holdout, holdout_probs)
    auc_train        = roc_auc_score(Y_train,   train_probs)

    # CDC HHI baseline (from published literature)
    HHI_AUC = 0.73

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('SustAInable — ROC Curve Analysis', fontsize=13, fontweight='bold')

    # ── Full ROC: train vs holdout vs baseline ─────────────────────────────────
    ax = axes[0]
    ax.plot(fpr_h, tpr_h, color='#e74c3c', linewidth=2.5,
            label=f'Holdout (AUC = {auc_holdout:.4f})')
    ax.plot(fpr_t, tpr_t, color='#3498db', linewidth=1.5, linestyle='--',
            label=f'Training (AUC = {auc_train:.4f})', alpha=0.7)
    ax.plot([0, 1], [0, 1], color='#95a5a6', linestyle=':', linewidth=1.5,
            label='Random classifier (AUC = 0.50)')
    # HHI baseline band
    ax.axhline(HHI_AUC, color='#f39c12', linestyle='-.', linewidth=1.5, alpha=0.8,
               label=f'CDC HHI baseline (AUC ≈ {HHI_AUC})')
    ax.fill_between([0, 1], [0, HHI_AUC], color='#f39c12', alpha=0.05)
    # Target region
    ax.axhline(0.85, color='#2ecc71', linestyle=':', linewidth=1, alpha=0.6,
               label='AUC target (0.85)')
    # Operating point
    idx = np.argmin(np.abs(
        np.interp(np.arange(0, 1.01, 0.01), fpr_h, tpr_h) -
        recall_score(Y_holdout, holdout_preds, zero_division=0)
    ))
    op_fpr = float(np.interp(
        recall_score(Y_holdout, holdout_preds, zero_division=0), tpr_h[::-1], fpr_h[::-1]
    ))
    op_tpr = recall_score(Y_holdout, holdout_preds, zero_division=0)
    ax.scatter([op_fpr], [op_tpr], color='#e74c3c', s=120, zorder=5,
               label=f'Operating point (threshold={FINAL_THRESHOLD:.2f})')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate (Recall)')
    ax.set_title('ROC Curve — Holdout Set (2022–2024 Events)')
    ax.legend(fontsize=9, loc='lower right')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

    # ── Zoomed view of the high-performance region ─────────────────────────────
    ax2 = axes[1]
    ax2.plot(fpr_h, tpr_h, color='#e74c3c', linewidth=2.5,
             label=f'Holdout (AUC = {auc_holdout:.4f})')
    ax2.scatter([op_fpr], [op_tpr], color='#e74c3c', s=120, zorder=5)
    ax2.axhline(0.80, color='#e74c3c', linestyle=':', linewidth=1.5, alpha=0.7,
                label='Recall target (0.80)')
    ax2.axhline(HHI_AUC, color='#f39c12', linestyle='-.', linewidth=1.5,
                label=f'HHI baseline TPR ≈ {HHI_AUC}')
    ax2.set_xlim(-0.01, 0.5)
    ax2.set_ylim(0.4, 1.02)
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate (Recall)')
    ax2.set_title('Zoomed — Low FPR Region')
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '05a_roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Summary ───────────────────────────────────────────────────────────────
    improvement = ((auc_holdout - HHI_AUC) / HHI_AUC) * 100
    print(f"✅ Chart saved to outputs/05a_roc_curve.png")
    print()
    print(f"AUC-ROC Summary:")
    print(f"  SustAInable (holdout): {auc_holdout:.4f}")
    print(f"  CDC HHI baseline:      {HHI_AUC:.4f}")
    if auc_holdout > HHI_AUC:
        print(f"  ✅ SustAInable outperforms baseline by {improvement:.1f}%")
    else:
        print(f"  ⚠️  Below baseline — review model config in Notebook 04")
else:
    print("⚠️  ROC curve skipped — model not scored.")

## Step 4: Confusion Matrix with Cost Analysis

The confusion matrix tells the complete story of how the model is performing — but raw numbers don't capture the asymmetric cost. This cell builds an annotated confusion matrix that shows the human consequence of each quadrant, not just the count.

In [ ]:
if holdout_preds is not None:
    cm = confusion_matrix(Y_holdout, holdout_preds)
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2,2)
                      else (cm[0,0], 0, 0, cm[1,1]))

    total = tn + fp + fn + tp

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'SustAInable — Confusion Matrix (Holdout, threshold={FINAL_THRESHOLD:.2f})',
                 fontsize=13, fontweight='bold')

    # ── Standard heatmap ──────────────────────────────────────────────────────
    cm_pct = cm.astype(float) / total * 100
    sns.heatmap(cm, annot=False, fmt='d', ax=axes[0],
                cmap='RdYlGn', linewidths=2,
                xticklabels=['Predicted 0\n(Not Elevated)', 'Predicted 1\n(Elevated)'],
                yticklabels=['Actual 0\n(Not Elevated)', 'Actual 1\n(Elevated)'])

    # Custom annotations with counts and percentages
    labels = [[f'{tn:,}\n({tn/total*100:.1f}%)', f'{fp:,}\n({fp/total*100:.1f}%)'],
              [f'{fn:,}\n({fn/total*100:.1f}%)', f'{tp:,}\n({tp/total*100:.1f}%)']]
    for i in range(2):
        for j in range(2):
            axes[0].text(j+0.5, i+0.5, labels[i][j],
                        ha='center', va='center', fontsize=12, fontweight='bold',
                        color='white' if (i==0 and j==1) or (i==1 and j==0) else 'black')

    axes[0].set_title('Counts and Percentages')
    axes[0].set_ylabel('Actual Label')
    axes[0].set_xlabel('Predicted Label')

    # ── Cost analysis panel ────────────────────────────────────────────────────
    ax2 = axes[1]
    ax2.axis('off')

    cost_data = [
        ("TRUE NEGATIVE", f"{tn:,}", f"{tn/total*100:.1f}%",
         "ZCTA correctly not flagged.
No resources deployed.
No harm done.",
         "#2ecc71", "✅"),
        ("FALSE POSITIVE", f"{fp:,}", f"{fp/total*100:.1f}%",
         "ZCTA flagged; no elevated illness.
Cooling resources deployed unnecessarily.
~$2,000–$8,000 per ZCTA. Recoverable.",
         "#f39c12", "⚠️"),
        ("FALSE NEGATIVE", f"{fn:,}", f"{fn/total*100:.1f}%",
         "ZCTA not flagged; elevated illness occurred.
Disabled/elderly residents not reached.
Preventable hospitalizations. NOT recoverable.",
         "#e74c3c", "🔴"),
        ("TRUE POSITIVE", f"{tp:,}", f"{tp/total*100:.1f}%",
         "ZCTA correctly flagged.
Cooling resources deployed in time.
Lives protected.",
         "#27ae60", "✅"),
    ]

    y_pos = 0.95
    for name, count, pct, consequence, color, icon in cost_data:
        ax2.text(0.02, y_pos, f"{icon} {name}", fontsize=11, fontweight='bold',
                 color=color, transform=ax2.transAxes, va='top')
        ax2.text(0.55, y_pos, f"{count}  ({pct})", fontsize=11, fontweight='bold',
                 color=color, transform=ax2.transAxes, va='top')
        y_pos -= 0.055
        for line in consequence.split('
'):
            ax2.text(0.04, y_pos, line, fontsize=9, color='#2c3e50',
                     transform=ax2.transAxes, va='top')
            y_pos -= 0.04
        y_pos -= 0.02
        ax2.axhline(y_pos + 0.01, color='#ecf0f1', linewidth=1,
                    transform=ax2.transAxes)

    ax2.set_title('Human Cost by Quadrant', fontsize=11, fontweight='bold', pad=10)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '05b_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/05b_confusion_matrix.png")

    # ── Metrics table ─────────────────────────────────────────────────────────
    print()
    print("=" * 58)
    print(f"CLASSIFICATION REPORT  (holdout, threshold = {FINAL_THRESHOLD:.2f})")
    print("=" * 58)
    metrics_table = [
        ("Recall (sensitivity)", recall_score(Y_holdout, holdout_preds, zero_division=0), 0.80),
        ("Precision",            precision_score(Y_holdout, holdout_preds, zero_division=0), 0.40),
        ("F1 Score",             f1_score(Y_holdout, holdout_preds, zero_division=0), None),
        ("AUC-ROC",              roc_auc_score(Y_holdout, holdout_probs), 0.85),
        ("PR-AUC",               average_precision_score(Y_holdout, holdout_probs), None),
        ("Brier Score",          brier_score_loss(Y_holdout, holdout_probs), None),
    ]
    for name, val, target in metrics_table:
        if target:
            flag = "✅" if val >= target else "⚠️ "
            tstr = f"  [target ≥ {target}]"
        else:
            flag, tstr = "  ", ""
        print(f"  {flag} {name:<25} {val:.4f}{tstr}")
else:
    print("⚠️  Confusion matrix skipped — model not scored.")

## Step 5: Score Distribution and Risk Tiers

This is the chart that matters most to an operations team. It shows how the model's probability scores distribute across the three risk tiers, and whether label=1 ZCTAs are actually clustering at the top of the score range — which is what we want to see.

In [ ]:
if holdout_df is not None:
    # ── Assign risk tiers ──────────────────────────────────────────────────────
    def assign_tier(score):
        if score >= 0.60:   return 'High (≥0.60)'
        elif score >= 0.30: return 'Elevated (0.30–0.59)'
        else:               return 'Baseline (<0.30)'

    holdout_df['risk_tier'] = holdout_df['prob_score'].apply(assign_tier)
    tier_order = ['High (≥0.60)', 'Elevated (0.30–0.59)', 'Baseline (<0.30)']
    tier_colors = {'High (≥0.60)': '#e74c3c',
                   'Elevated (0.30–0.59)': '#f39c12',
                   'Baseline (<0.30)': '#3498db'}

    fig = plt.figure(figsize=(18, 6))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
    fig.suptitle('SustAInable — Score Distribution and Risk Tiers (Holdout Set)',
                 fontsize=13, fontweight='bold')

    # 1. Score histogram by true label
    ax1 = fig.add_subplot(gs[0])
    scores_pos = holdout_df[holdout_df['label'] == 1]['prob_score']
    scores_neg = holdout_df[holdout_df['label'] == 0]['prob_score']
    bins = np.linspace(0, 1, 41)
    ax1.hist(scores_neg, bins=bins, color='#3498db', alpha=0.7, label='Label = 0 (not elevated)')
    ax1.hist(scores_pos, bins=bins, color='#e74c3c', alpha=0.7, label='Label = 1 (elevated)')
    ax1.axvline(FINAL_THRESHOLD, color='#2c3e50', linestyle='--', linewidth=2,
                label=f'Threshold ({FINAL_THRESHOLD:.2f})')
    ax1.axvline(0.60, color='#e74c3c', linestyle=':', linewidth=1.5, alpha=0.7,
                label='High tier (0.60)')
    ax1.set_xlabel('Probability Score')
    ax1.set_ylabel('ZCTA-Event Count')
    ax1.set_title('Score Distribution by True Label')
    ax1.legend(fontsize=8)

    # 2. Risk tier composition
    ax2 = fig.add_subplot(gs[1])
    tier_counts = holdout_df['risk_tier'].value_counts()
    tier_counts = tier_counts.reindex([t for t in tier_order if t in tier_counts.index])
    colors_list = [tier_colors[t] for t in tier_counts.index]
    bars = ax2.bar(range(len(tier_counts)), tier_counts.values,
                   color=colors_list, edgecolor='white')
    ax2.set_xticks(range(len(tier_counts)))
    ax2.set_xticklabels(
        [t.replace(' (', '
(') for t in tier_counts.index],
        fontsize=8
    )
    ax2.set_ylabel('ZCTA-Event Count')
    ax2.set_title('ZCTAs per Risk Tier')
    total_h = len(holdout_df)
    for bar, val in zip(bars, tier_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:,}
({val/total_h*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

    # 3. Positive rate within each tier
    ax3 = fig.add_subplot(gs[2])
    tier_pos_rates = (
        holdout_df.groupby('risk_tier')['label']
        .agg(['mean', 'sum', 'count'])
        .rename(columns={'mean': 'positive_rate', 'sum': 'positives', 'count': 'total'})
        .reindex([t for t in tier_order if t in holdout_df['risk_tier'].unique()])
    )
    bar_colors_rate = [tier_colors[t] for t in tier_pos_rates.index]
    bars3 = ax3.bar(range(len(tier_pos_rates)),
                    tier_pos_rates['positive_rate'] * 100,
                    color=bar_colors_rate, edgecolor='white')
    ax3.set_xticks(range(len(tier_pos_rates)))
    ax3.set_xticklabels(
        [t.replace(' (', '
(') for t in tier_pos_rates.index],
        fontsize=8
    )
    ax3.set_ylabel('% of ZCTAs with Label = 1')
    ax3.set_title('Positive Rate by Risk Tier')
    for bar, (_, row) in zip(bars3, tier_pos_rates.iterrows()):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{row.positive_rate*100:.1f}%
({int(row.positives):,}/{int(row.total):,})',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.savefig(OUTPUTS / '05c_score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/05c_score_distribution.png")

    # ── Tier summary table ─────────────────────────────────────────────────────
    print()
    print("Risk Tier Summary (Holdout Set):")
    display(tier_pos_rates.round(4))
else:
    print("⚠️  Score distribution skipped — holdout data not scored.")

## Step 6: Per-Event Performance Breakdown

This is the analysis that would matter most to a transit agency or public health department evaluating SustAInable for a pilot. It answers: *does the model work consistently across different types of heat events, or does it only perform well on certain kinds?*

Good generalization means similar performance across regional events, short-duration events, and long-duration events.

In [ ]:
if holdout_df is not None and 'event_id' in holdout_df.columns:
    event_metrics = []
    for event_id, group in holdout_df.groupby('event_id'):
        if len(group) < 10:
            continue
        y_true  = group['label']
        y_prob  = group['prob_score']
        y_pred  = group['prediction']
        n_pos   = y_true.sum()
        n_total = len(group)

        if n_pos == 0 or n_pos == n_total:
            auc = None
        else:
            try:
                auc = roc_auc_score(y_true, y_prob)
            except Exception:
                auc = None

        event_metrics.append({
            'event_id':     event_id,
            'event_label':  group['event_label'].iloc[0] if 'event_label' in group.columns else event_id,
            'n_zctas':      n_total,
            'n_elevated':   int(n_pos),
            'pos_rate':     round(float(n_pos / n_total), 4),
            'recall':       round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
            'precision':    round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
            'f1':           round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
            'auc_roc':      round(float(auc), 4) if auc is not None else None,
        })

    event_df = pd.DataFrame(event_metrics)
    print("Per-Event Performance (Holdout Set):")
    display(event_df.drop(columns=['event_id']))

    if len(event_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(event_df) * 0.6 + 2)))
        fig.suptitle('SustAInable — Per-Event Performance (Holdout Set)',
                     fontsize=13, fontweight='bold')

        short_labels = [e.replace('HE_', '').replace('_', ' ')
                        for e in event_df['event_id']]

        # Recall per event
        recall_colors = ['#2ecc71' if r >= 0.80 else '#e74c3c'
                         for r in event_df['recall']]
        axes[0].barh(short_labels, event_df['recall'],
                     color=recall_colors, edgecolor='white', alpha=0.85)
        axes[0].axvline(0.80, color='#2c3e50', linestyle='--', linewidth=2,
                        label='Target recall (0.80)')
        axes[0].set_xlabel('Recall')
        axes[0].set_title('Recall by Event
(green = meets target)')
        axes[0].legend(fontsize=9)
        axes[0].set_xlim(0, 1.05)
        for i, val in enumerate(event_df['recall']):
            axes[0].text(val + 0.01, i, f'{val:.2f}', va='center', fontsize=9)

        # AUC per event
        auc_vals    = event_df['auc_roc'].fillna(0)
        auc_colors  = ['#2ecc71' if a >= 0.85 else ('#f39c12' if a >= 0.73 else '#e74c3c')
                       for a in auc_vals]
        axes[1].barh(short_labels, auc_vals,
                     color=auc_colors, edgecolor='white', alpha=0.85)
        axes[1].axvline(0.85, color='#2c3e50', linestyle='--', linewidth=2,
                        label='AUC target (0.85)')
        axes[1].axvline(0.73, color='#f39c12', linestyle=':', linewidth=1.5,
                        label='HHI baseline (0.73)')
        axes[1].set_xlabel('AUC-ROC')
        axes[1].set_title('AUC-ROC by Event
(green=target, yellow=beats baseline)')
        axes[1].legend(fontsize=8)
        axes[1].set_xlim(0, 1.05)
        for i, val in enumerate(auc_vals):
            label_str = f'{val:.2f}' if val > 0 else 'N/A'
            axes[1].text(val + 0.01, i, label_str, va='center', fontsize=9)

        plt.tight_layout()
        plt.savefig(OUTPUTS / '05d_per_event_performance.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("\n✅ Chart saved to outputs/05d_per_event_performance.png")

        # Summary stats
        valid_auc = event_df['auc_roc'].dropna()
        print()
        print(f"Events meeting recall target  (≥0.80): "
              f"{(event_df['recall'] >= 0.80).sum()} / {len(event_df)}")
        if len(valid_auc) > 0:
            print(f"Events beating HHI baseline   (≥0.73): "
                  f"{(valid_auc >= 0.73).sum()} / {len(valid_auc)}")
            print(f"Events meeting AUC target     (≥0.85): "
                  f"{(valid_auc >= 0.85).sum()} / {len(valid_auc)}")
else:
    print("⚠️  Per-event analysis skipped — holdout data not available.")

## Step 7: Calibration Curve

Calibration tells us whether the model's probability scores are meaningful — not just whether the rankings are correct. A well-calibrated model that gives a score of 0.70 means "70% of similarly-scored ZCTAs experienced elevated illness."

This matters for stakeholders. If a public health director asks "how confident are you that ZIP code 19132 will see elevated illness?" — the probability score is only a useful answer if the model is well calibrated.

A perfectly calibrated model follows the diagonal line.

In [ ]:
if holdout_probs is not None and len(np.unique(Y_holdout)) > 1:
    try:
        # Use 8 bins for calibration (more stable with moderate sample sizes)
        n_bins = min(8, int(len(Y_holdout) / 20))
        n_bins = max(n_bins, 3)

        fraction_pos, mean_pred = calibration_curve(
            Y_holdout, holdout_probs, n_bins=n_bins, strategy='quantile'
        )

        brier = brier_score_loss(Y_holdout, holdout_probs)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('SustAInable — Probability Calibration (Holdout Set)',
                     fontsize=13, fontweight='bold')

        # Calibration curve
        axes[0].plot([0, 1], [0, 1], color='#95a5a6', linestyle='--',
                     linewidth=1.5, label='Perfect calibration')
        axes[0].plot(mean_pred, fraction_pos, color='#e74c3c', linewidth=2.5,
                     marker='o', markersize=7, label=f'SustAInable (Brier={brier:.4f})')
        axes[0].fill_between(mean_pred, fraction_pos, mean_pred,
                             alpha=0.15, color='#e74c3c', label='Calibration gap')
        axes[0].set_xlabel('Mean Predicted Probability')
        axes[0].set_ylabel('Fraction of Positives (Actual)')
        axes[0].set_title('Calibration Curve')
        axes[0].legend(fontsize=9)
        axes[0].set_xlim(-0.02, 1.02)
        axes[0].set_ylim(-0.02, 1.02)

        # Score histogram below the curve
        axes[1].hist(holdout_probs[Y_holdout == 0], bins=40,
                     color='#3498db', alpha=0.7, label='Label = 0', density=True)
        axes[1].hist(holdout_probs[Y_holdout == 1], bins=40,
                     color='#e74c3c', alpha=0.7, label='Label = 1', density=True)
        axes[1].axvline(FINAL_THRESHOLD, color='#2c3e50', linestyle='--',
                        linewidth=2, label=f'Threshold ({FINAL_THRESHOLD:.2f})')
        axes[1].set_xlabel('Predicted Probability')
        axes[1].set_ylabel('Density')
        axes[1].set_title('Score Density by True Label')
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.savefig(OUTPUTS / '05e_calibration_curve.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✅ Chart saved to outputs/05e_calibration_curve.png")

        # Brier score interpretation
        print()
        print(f"Brier Score: {brier:.4f}")
        print("  (0.00 = perfect | 0.25 = no skill | lower is better)")
        if brier < 0.10:
            print("  ✅ Excellent calibration")
        elif brier < 0.15:
            print("  ✅ Good calibration")
        elif brier < 0.20:
            print("  ⚠️  Moderate calibration — consider Platt scaling in Notebook 06")
        else:
            print("  ⚠️  Poor calibration — probability scores may not be reliable")
            print("  Consider isotonic regression or Platt scaling post-training")

    except Exception as e:
        print(f"⚠️  Calibration curve error: {e}")
        print("   This can happen with very small holdout sets.")
        print("   It will resolve with more data.")
else:
    print("⚠️  Calibration skipped — insufficient holdout data.")

## Step 8: Full Evaluation Summary

The single-page summary you would include in a fellowship application, pitch deck, or partner presentation.

In [ ]:
if holdout_probs is not None:
    h_auc   = roc_auc_score(Y_holdout, holdout_probs)
    h_prauc = average_precision_score(Y_holdout, holdout_probs)
    h_rec   = recall_score(Y_holdout, holdout_preds, zero_division=0)
    h_prec  = precision_score(Y_holdout, holdout_preds, zero_division=0)
    h_f1    = f1_score(Y_holdout, holdout_preds, zero_division=0)
    h_brier = brier_score_loss(Y_holdout, holdout_probs)
    cm      = confusion_matrix(Y_holdout, holdout_preds)
    tn, fp, fn, tp = (cm.ravel() if cm.shape == (2,2) else (0, 0, 0, 0))
    HHI_AUC = 0.73

    proxy_note = " ⚠️  PROXY LABELS" if IS_PROXY else ""

    print("=" * 62)
    print(f"SUSTAINAINABLE — FULL EVALUATION SUMMARY{proxy_note}")
    print("=" * 62)
    print(f"  Holdout events:  2022–2024 (never seen during training)")
    print(f"  Decision threshold: {FINAL_THRESHOLD:.2f}  (asymmetric cost design)")
    print()
    print("PERFORMANCE METRICS")
    print("-" * 62)
    rows = [
        ("AUC-ROC",              h_auc,   0.85, True),
        ("PR-AUC",               h_prauc, None, False),
        ("Recall (sensitivity)", h_rec,   0.80, True),
        ("Precision",            h_prec,  0.40, True),
        ("F1 Score",             h_f1,    None, False),
        ("Brier Score",          h_brier, None, False),
    ]
    for name, val, target, has_target in rows:
        if has_target and target:
            flag = "✅" if val >= target else "⚠️ "
            tstr = f"  target ≥ {target}"
        else:
            flag, tstr = "  ", ""
        print(f"  {flag} {name:<25} {val:.4f}{tstr}")

    print()
    print("BASELINE COMPARISON")
    print("-" * 62)
    improvement = ((h_auc - HHI_AUC) / HHI_AUC) * 100
    print(f"  CDC HHI standalone AUC:    {HHI_AUC:.4f}")
    print(f"  SustAInable AUC:           {h_auc:.4f}")
    beat = "✅ Outperforms" if h_auc > HHI_AUC else "⚠️  Below"
    print(f"  Result: {beat} baseline by {abs(improvement):.1f}%")

    print()
    print("CONFUSION MATRIX")
    print("-" * 62)
    print(f"  True Negatives  (safe ZCTAs correctly cleared): {tn:>7,}")
    print(f"  False Positives (unnecessary resource deploy):  {fp:>7,}  ← recoverable")
    print(f"  False Negatives (elevated illness missed):      {fn:>7,}  ← NOT recoverable")
    print(f"  True Positives  (elevated illness caught):      {tp:>7,}")

    if fn + tp > 0:
        miss_rate   = fn / (fn + tp)
        capture_rate = tp / (fn + tp)
        print()
        print(f"  Capture rate (% of elevated ZCTAs flagged): {capture_rate:.1%}")
        print(f"  Miss rate   (% of elevated ZCTAs missed):   {miss_rate:.1%}")

    print()
    print("OUTPUTS SAVED")
    print("-" * 62)
    for fname in ['05a_roc_curve.png', '05b_confusion_matrix.png',
                  '05c_score_distribution.png', '05d_per_event_performance.png',
                  '05e_calibration_curve.png']:
        path = OUTPUTS / fname
        exists = "✅" if path.exists() else "⚠️  not yet created"
        print(f"  {exists}  {fname}")
else:
    print("⚠️  Summary skipped — model not scored.")

---

## ✅ What You Just Built

A complete, publication-ready evaluation of the SustAInable model across five dimensions: discrimination (ROC), error type analysis (confusion matrix), score distribution and risk tiers, per-event consistency, and probability calibration.

Every chart in this notebook is fellowship-ready.

---

## 🔜 What Comes Next

| Notebook | Purpose |
|---|---|
| `06_shap_explainability.ipynb` | SHAP values — *why* each ZIP code was flagged, per-ZCTA feature attribution |

SHAP is what turns SustAInable from a black box into a tool a community organization can actually challenge and trust — which is essential for equitable deployment.

---

## ⚠️ Proxy Label Note

If you saw proxy label warnings throughout this notebook, the pipeline is correct — the model and evaluation architecture are production-ready. The numbers will sharpen significantly when NSSP data replaces the proxy labels.

**NSSP access:** https://www.cdc.gov/nssp/php/onboarding/index.html

---

*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*